## Projeto Criação de Dados fictícios com Python

#### Sobre

Este projeto tem como objetivo demonstrar como utilizar Python para gerar dados sintéticos que possam ser utilizados na criação e no teste de bancos de dados.

Neste notebook será criado um pequeno conjunto de dados composto por duas tabelas relacionadas: clientes e compras. Essas tabelas representam um cenário simples de um sistema de vendas, onde clientes realizam pedidos ao longo do tempo.

<b>Tabela: clientes</b>

A tabela clientes representa as pessoas cadastradas no sistema. Cada registro corresponde a um cliente único que pode realizar uma ou mais compras. Além das informações básicas de identificação, também será armazenada a data em que o cliente foi registrado no sistema.
<br><br>
Todo Cliente possui nome completo, cidade em que reside, email, número de celular e data de registro.

<b>Tabela: compras</b>

A tabela compras representa as transações realizadas pelos clientes. Cada registro corresponde a uma compra efetuada por um cliente específico. Essa tabela possui uma relação direta com a tabela de clientes por meio do campo cliente_id, que identifica qual cliente realizou a compra.

#### Bibliotecas

In [1]:
import sqlite3
import random
from datetime import datetime, timedelta

#### Definindo informações

In [24]:
nomes = [
"Lucas","Mariana","Pedro","Ana","Gabriel","Julia","Rafael","Beatriz", "Matheus","Carolina","Thiago",
    "Larissa","Bruno","Camila","Felipe","Ricardo","Fernanda","Eduardo","Patricia","Daniel"
]

sobrenomes = [
"Silva","Santos","Oliveira","Souza","Pereira","Rodrigues","Almeida", "Nascimento","Lima","Araújo",
    "Fernandes","Carvalho","Gomes","Martins","Rocha","Barbosa","Dias","Teixeira","Correia","Monteiro"
]

cidades = [
"Porto Alegre","São Paulo","Rio de Janeiro","Curitiba","Florianópolis", "Belo Horizonte","Brasília",
    "Salvador","Recife","Fortaleza","Campinas","Caxias do Sul","Pelotas","Joinville","Londrina"
]

dominios_email = [
"gmail.com","hotmail.com","outlook.com","yahoo.com","icloud.com","protonmail.com","bol.com.br"
]

metodos_pagamento = [
    "Pix",
    "Cartão de Crédito",
    "Cartão de Débito"
]

produtos_catalogo = {
    "Camisetas": [
        "Camiseta Básica",
        "Camiseta Estampada",
        "Camiseta Oversized",
        "Polo Masculina",
        "Polo Feminina"
    ],
    
    "Calcas": [
        "Calça Jeans Slim",
        "Calça Jeans Reta",
        "Calça Moletom",
        "Calça Cargo"
    ],

    "Tenis": [
        {"produto": "Tênis Street Classic", "linha": "para o dia a dia"},
        {"produto": "Tênis Urban Style", "linha": "para o dia a dia"},
        {"produto": "Tênis Daily Comfort", "linha": "para o dia a dia"},
        
        {"produto": "Tênis Skate Pro", "linha": "para skate"},
        {"produto": "Tênis Skate Street", "linha": "para skate"},
        {"produto": "Tênis Skate Vulcan", "linha": "para skate"},
        
        {"produto": "Tênis Eco Run", "linha": "sem origem animal"},
        {"produto": "Tênis Vegan Street", "linha": "sem origem animal"},
        {"produto": "Tênis Eco Casual", "linha": "sem origem animal"}
    ]
    
}

#### Gerando Clientes

In [3]:
# Função para gerar nomes
def gerar_nomes():
    return f"{random.choice(nomes)} {random.choice(sobrenomes)}"

In [4]:
# Função para gerar email
def gerar_emails(nome):
    nome_email = nome.lower().replace(" ", ".")
    return f"{nome_email}@{random.choice(dominios_email)}"

In [5]:
# Função para gerar número de celular
def gerar_celular():
    return f"({random.randint(10,99)}) {random.randint(90000, 99999)}-{random.randint(1000, 9999)}"

In [6]:
# Função para gerar cidade
def gerar_cidades():
    return random.choice(cidades)

In [63]:
# Função para gerar data de cadastro
def gerar_datas(data=None):
    if data is None:
        data = datetime(2023,1,1)
    if isinstance(data, str):
        data = datetime.strptime(data, "%Y-%m-%d")
    fim = datetime.now()
    dif = fim - data
    return (data + timedelta(days=random.randint(0, dif.days))).strftime("%Y-%m-%d")

In [64]:
clientes = []

for i in range(1, 101):
    nome = gerar_nomes()
    cliente = {
        "id":i,
        "nome": nome,
        "email": gerar_emails(nome),
        "cidade": gerar_cidades(),
        "celular": gerar_celular(),
        "cidade": gerar_cidades(),
        "data_registro": gerar_datas()
    }
    clientes.append(cliente)

#### Gerando Pedidos

In [66]:
def pegar_produto():
    categoria = random.choice(list(produtos_catalogo.keys()))
    if categoria == "Tenis":
        item = random.choice(produtos_catalogo[categoria])
        produto = item["produto"]
        linha = item["linha"]
    else:
        produto = random.choice(produtos_catalogo[categoria])
        linha = None
    return {
        "categoria": categoria,
        "produto": produto,
        "linha": linha
    }

In [67]:
pegar_produto()

{'categoria': 'Tenis', 'produto': 'Tênis Skate Pro', 'linha': 'para skate'}

In [68]:
def gerar_pedido(clientes):
    cliente = random.choice(clientes)
    cliente_id = cliente["id"]
    cliente_data_registro = cliente["data_registro"]

    item = pegar_produto()
    if item["categoria"] == "Camisetas" or "Calcas":
        valor = random.uniform(90, 300)
    else:
        valor = random.uniform(250, 750)

    metodo_pagamento = random.choice(metodos_pagamento)

    data_pedido = gerar_datas(cliente_data_registro)
    
    return {
        "cliente_id": cliente_id,
        "produto": item["produto"],
        "categoria": item["categoria"],
        "linha": item["linha"],
        "valor": round(valor, 2),
        "metodo_pagamento": metodo_pagamento,
        "data_pedido": data_pedido
    }

In [70]:
pedidos = []

for i in range(500):
    pedido = gerar_pedido(clientes)
    pedidos.append(pedido)

#### Preparando o BD

In [72]:
conn = sqlite3.connect("loja.db")
cursor = conn.cursor()

In [73]:
cursor.execute(
    """
        CREATE TABLE IF NOT EXISTS clientes (
            id INTEGER PRIMARY KEY,
            nome TEXT,
            email TEXT,
            cidade TEXT,
            celular TEXT,
            data_registro DATE
        )
    """
)

In [74]:
cursor.execute(
    """
        CREATE TABLE IF NOT EXISTS pedidos (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            cliente_id INTEGER,
            produto TEXT,
            categoria TEXT,
            linha TEXT,
            valor REAL,
            metodo_pagamento TEXT,
            data_compra DATE,
            FOREIGN KEY (cliente_id) REFERENCES clientes(id)
        )
    """
)

#### Inserindo Dados no DB

In [75]:
for c in clientes:
    cursor.execute(
        """
            INSERT INTO clientes (id, nome, email, cidade, celular, data_registro)
            VALUES (?, ?, ?, ?, ?, ?)
        """, (
            c["id"],
            c["nome"],
            c["email"],
            c["cidade"],
            c["celular"],
            c["data_registro"]
        )
    )

In [77]:
for p in pedidos:
    cursor.execute(
        """
            INSERT INTO pedidos (cliente_id, produto, categoria, linha, valor, metodo_pagamento, data_compra)
            VALUES (?, ?, ?, ?, ?, ?, ?)
        """, (
            p["cliente_id"],
            p["produto"],
            p["categoria"],
            p["linha"],
            p["valor"],
            p["metodo_pagamento"],
            p["data_pedido"]
        )
    )

In [78]:
conn.commit()
conn.close()